In [265]:
import pandas as pd
import numpy as np

movies = pd.read_csv("../data/TMDB  IMDB Movies Dataset.csv")

In [266]:
movies.head(1)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2817446,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."


In [267]:
movies.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'tconst', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='object')

In [268]:
df = movies.copy()

In [269]:
df.duplicated().sum()

np.int64(0)

In [270]:
df.dropna(subset = "title", inplace = True)
df.dropna(subset = "overview", inplace = True)
df.dropna(subset = "poster_path", inplace = True)
df.dropna(subset = "release_date", inplace = True)

df.drop_duplicates(subset = "id")
df.drop_duplicates(subset=["title", "release_date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 326953 entries, 0 to 437930
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    326953 non-null  int64  
 1   title                 326953 non-null  object 
 2   vote_average          326953 non-null  float64
 3   vote_count            326953 non-null  int64  
 4   status                326953 non-null  object 
 5   release_date          326953 non-null  object 
 6   revenue               326953 non-null  int64  
 7   runtime               326953 non-null  int64  
 8   adult                 326953 non-null  bool   
 9   backdrop_path         173172 non-null  object 
 10  budget                326953 non-null  int64  
 11  homepage              47441 non-null   object 
 12  tconst                326953 non-null  object 
 13  original_language     326953 non-null  object 
 14  original_title        326953 non-null  object 
 15  overv

In [271]:
df.shape

(326953, 29)

In [272]:
df['release_date'] = pd.to_datetime(df['release_date'], errors="coerce").dt.year
df = df[df['release_date'] >= 1975]

df.shape

(262013, 29)

In [273]:
df.head(1)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2817446,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."


In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()

    return text


def preprocess_text(texts):
    texts = [clean_text(x) for x in texts]

    docs = nlp.pipe(
        texts,
        batch_size=10000,
        n_process=-1
    )

    processed = []

    for doc in docs:
        processed.append(
            " ".join(
                token.text if token.pos_ == "PROPN" else token.lemma_
                for token in doc
                if not (
                    token.is_punct
                    or token.is_space
                )
            )
        )

    return processed


df["overview_text"] = preprocess_text(df["overview"].fillna("") + " " + df["tagline"].fillna(""))

df["keywords_text"] = preprocess_text(df["keywords"])

df["genre_text"] = preprocess_text(df["genres"])

df["people_text"] = preprocess_text(
    df["directors"].fillna("") + " " +
    df["writers"].fillna("") + " " +
    df["cast"].fillna("")
)

df["metadata_text"] = preprocess_text(
    df["production_companies"].fillna("") + " " +
    df["production_countries"].fillna("") + " " +
    df["spoken_languages"].fillna("") + " " +
    df["original_language"].fillna("")
)

print(df.columns)
df.head(1)

In [ ]:
final_df = df[
    [
        'id',
        'title',

        "overview",

        'overview_text',
        'keywords_text',
        'genre_text',
        'people_text',
        'metadata_text',

        'vote_average',
        'vote_count',
        'popularity',
        'runtime',

        'poster_path',
        'release_date'
    ]
]
final_df.head()

,id,title,overview_text,keywords_text,genre_text,metadata_text,vote_average,vote_count,popularity,runtime,poster_path,release_date
0,27205,Inception,cobb skilled thief commit corporate espionage ...,rescue mission dream airplane paris france vir...,action science fiction adventure,legendary picture syncopy warner bros pictures...,8.364,34495,83.952,148,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,2010
1,157336,Interstellar,adventure group explorer use newly discover wo...,rescue future spacecraft race time artificial ...,adventure drama science fiction,legendary picture syncopy lynda obst productio...,8.417,32571,140.241,169,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,2014
2,155,The Dark Knight,batman raise stake war crime help lt jim gordo...,joker sadism chaos secret identity crime fight...,drama action crime thriller,dc comic_strip legendary picture syncopy isobe...,8.512,30619,130.643,152,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,2008
3,19995,Avatar,22nd century paraplegic marine dispatch moon p...,future society culture clash space travel spac...,action adventure fantasy science fiction,dune entertainment lightstorm entertainment 20...,7.573,29815,79.932,162,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,2009
4,24428,The Avengers,unexpected enemy emerge threaten global safety...,new york city superhero shield base comic alie...,science fiction action adventure,marvel studio united states america english hi...,7.710,29166,98.082,143,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,2012
